# Stage 1 — Data Building Walkthrough



## Setup

In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
OUT_DATASET = DATA_DIR / "stage1.csv.gz"

# Names of the behavioural columns we'll produce. Useful as a checklist
# and to defensively guarantee column ordering at save time.
STAGE1_FEATURE_COLS = [
    # Event volume / shape (baseline — preprocess.ipynb)
    "total_events_0_2", "n_active_days_0_2",
    "n_click_events_0_2", "n_view_events_0_2",
    "n_sessions_0_2", "n_topics_event_0_2",
    "mean_hour_0_2",
    # Std twin 
    "std_hour_0_2",
    # Transactions (baseline)
    "total_transactions_0_2", "correct_rate_0_2", "partial_rate_0_2",
    "mean_evaluation_score_0_2",
    "avg_response_time_0_2",
    "n_documents_0_2", "n_topics_transaction_0_2",
    # Std twins 
    "std_evaluation_score_0_2", "std_response_time_0_2",
    # Session shape 
    "session_duration_mean_0_2", "session_duration_std_0_2",
    "time_between_sessions_mean_0_2",
    # engagement ratios (Beck & Gong 2013)
    "retry_ratio_0_2", "review_rate_0_2",
]
print(f"will produce {len(STAGE1_FEATURE_COLS)} behavioural features + a demographic block")

will produce 22 behavioural features + a demographic block


## 1. Load raw data

In [2]:
users = pd.read_csv(DATA_DIR / "users.csv.gz")
events = pd.read_csv(DATA_DIR / "events.csv.gz")
transactions = pd.read_csv(DATA_DIR / "transactions.csv.gz")

print(f"users:        {len(users):,} rows  /  {users['user_id'].nunique():,} unique users")
print(f"events:       {len(events):,} rows")
print(f"transactions: {len(transactions):,} rows")

users:        30,929 rows  /  30,929 unique users
events:       11,193,171 rows
transactions: 2,134,759 rows


## 2. Clean


### 2.1 Users — normalize categoricals


In [3]:
users_clean = users.copy()
users_clean["gender"] = (
    users_clean["gender"].replace({"*": np.nan, "STAR": "Other"}).fillna("Unknown")
)
users_clean["canton"] = users_clean["canton"].fillna("Unknown")
users_clean["class_level"] = users_clean["class_level"].fillna("Unknown")
users_clean["class_id"] = users_clean["class_id"].fillna("Unknown")
users_clean["study"] = (
    users_clean["study"]
    .replace({"True": True, "False": False})
    .fillna(False)
    .astype(bool)
)
del users
gc.collect()
print(f"users_clean: {len(users_clean):,} rows")
users_clean.head()

users_clean: 30,929 rows


,user_id,gender,canton,class_level,study,class_id
0,387604,Unknown,Unknown,Unknown,False,Unknown
1,387605,Unknown,Unknown,Unknown,False,Unknown
2,387608,Unknown,Unknown,Unknown,True,9Q2M7
3,387613,Unknown,Unknown,Unknown,False,Unknown
4,387615,MALE,SG,Gymnasium - 3. Jahr,False,Unknown


### 2.2 Events — parse timestamp and derive calendar features


In [4]:
events_clean = events.copy()
events_clean["event_date"] = pd.to_datetime(events_clean["event_date"], errors="coerce")
events_clean = events_clean.dropna(subset=["user_id", "event_date"])

events_clean["date"] = events_clean["event_date"].dt.date
events_clean["year"] = events_clean["event_date"].dt.year
events_clean["month"] = events_clean["event_date"].dt.month
events_clean["dayofweek"] = events_clean["event_date"].dt.dayofweek
events_clean["hour"] = events_clean["event_date"].dt.hour
events_clean["week"] = events_clean["event_date"].dt.to_period("W").astype(str)

del events
gc.collect()
print(f"events_clean: {len(events_clean):,} rows")
events_clean.head()

events_clean: 11,193,171 rows


,event_id,user_id,event_date,category,action,event_type,transaction_token,tracking_data,session_id,topic_id,session_closed,session_type,session_accepted,date,year,month,dayofweek,hour,week
0,62,393211,2021-05-21 07:56:54.885,TASK,VIEW_QUESTION,VIEW,7a10ca52-ffb5-4069-8800-0dc86d778e94,NaN,NaN,NaN,NaN,NaN,NaN,2021-05-21,2021,5,4,7,2021-05-17/2021-05-23
1,63,393211,2021-05-21 07:58:18.912,TASK,SUBMIT_ANSWER,CLICK,7a10ca52-ffb5-4069-8800-0dc86d778e94,NaN,NaN,NaN,NaN,NaN,NaN,2021-05-21,2021,5,4,7,2021-05-17/2021-05-23
2,64,393211,2021-05-21 07:58:27.207,TASK,NEXT,CLICK,7a10ca52-ffb5-4069-8800-0dc86d778e94,NaN,NaN,NaN,NaN,NaN,NaN,2021-05-21,2021,5,4,7,2021-05-17/2021-05-23
3,65,393211,2021-05-21 07:58:27.589,TASK,VIEW_QUESTION,VIEW,88fdcaad-f73b-46a2-b561-d262f2441442,NaN,NaN,NaN,NaN,NaN,NaN,2021-05-21,2021,5,4,7,2021-05-17/2021-05-23
4,66,393211,2021-05-21 08:03:42.588,TASK,SUBMIT_ANSWER,CLICK,88fdcaad-f73b-46a2-b561-d262f2441442,NaN,NaN,NaN,NaN,NaN,NaN,2021-05-21,2021,5,4,8,2021-05-17/2021-05-23


### 2.3 Transactions — parse timestamps + derive evaluation outcomes


In [5]:
transactions_clean = transactions.copy()
transactions_clean["start_time"] = pd.to_datetime(transactions_clean["start_time"], errors="coerce")
transactions_clean["commit_time"] = pd.to_datetime(transactions_clean["commit_time"], errors="coerce")
transactions_clean = transactions_clean.dropna(subset=["user_id", "start_time"])

transactions_clean["date"] = transactions_clean["start_time"].dt.date
transactions_clean["year"] = transactions_clean["start_time"].dt.year
transactions_clean["month"] = transactions_clean["start_time"].dt.month
transactions_clean["dayofweek"] = transactions_clean["start_time"].dt.dayofweek
transactions_clean["hour"] = transactions_clean["start_time"].dt.hour
transactions_clean["week"] = transactions_clean["start_time"].dt.to_period("W").astype(str)

transactions_clean["evaluation"] = transactions_clean["evaluation"].fillna("UNKNOWN")
transactions_clean["evaluation_score"] = (
    transactions_clean["evaluation"]
    .map({"CORRECT": 1.0, "PARTIAL": 0.5, "WRONG": 0.0, "UNKNOWN": 0.0})
    .fillna(0.0)
)
transactions_clean["is_correct"] = (transactions_clean["evaluation"] == "CORRECT").astype(int)
transactions_clean["is_partial"] = (transactions_clean["evaluation"] == "PARTIAL").astype(int)
transactions_clean["is_unknown_eval"] = (transactions_clean["evaluation"] == "UNKNOWN").astype(int)

transactions_clean["response_time_sec"] = (
    transactions_clean["commit_time"] - transactions_clean["start_time"]
).dt.total_seconds()
# Discard negative or implausibly long response times (> 1 h).
transactions_clean.loc[
    (transactions_clean["response_time_sec"] < 0) | (transactions_clean["response_time_sec"] > 3600),
    "response_time_sec",
] = np.nan

del transactions
gc.collect()
print(f"transactions_clean: {len(transactions_clean):,} rows")
transactions_clean.head()

transactions_clean: 2,134,759 rows


,transaction_id,transaction_token,user_id,document_id,document_version,evaluation,input,start_time,commit_time,user_agent,...,year,month,dayofweek,hour,week,evaluation_score,is_correct,is_partial,is_unknown_eval,response_time_sec
0,688413,88fdcaad-f73b-46a2-b561-d262f2441442,393211,awd0i1DlVtg6kuMZSkpmHa,75002,PARTIAL,"{""type"": ""MULTI_COLOR_HIGHLIGHT"", ""highlighted...",2021-05-21 07:58:27.312,2021-05-21 08:03:43.020000000,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,2021,5,4,7,2021-05-17/2021-05-23,0.5,0,1,0,315.708
1,688414,a75eb7b4-b2c2-47d4-9200-27980c175037,393211,arhWF3BT53V9W8cGOaZVPX,75012,PARTIAL,"{""type"": ""MULTI_COLOR_HIGHLIGHT"", ""highlighted...",2021-05-21 08:04:05.067,2021-05-21 08:07:21.288999936,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,2021,5,4,8,2021-05-17/2021-05-23,0.5,0,1,0,196.222
2,688415,61eb829d-bdda-4107-86af-ad9a14a7bdc9,393211,9wk5dtV2mF59odW0wCEYYc,75003,PARTIAL,"{""type"": ""CLOZE_TEXT"", ""clozeInputs"": [""Person...",2021-05-21 08:07:37.048,2021-05-21 08:13:30.953999872,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,2021,5,4,8,2021-05-17/2021-05-23,0.5,0,1,0,353.906
3,688416,30ff0d8a-865d-460b-9177-b698a52b0d5c,393211,afilxZ8LycP5LReULeKngW,75009,CORRECT,"{""type"": ""DND_PAIRS"", ""input"": [""<p>Ich gehe i...",2021-05-21 08:13:38.943,2021-05-21 08:22:13.975000064,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,2021,5,4,8,2021-05-17/2021-05-23,1.0,1,0,0,515.032
4,688417,0adedf3b-ba35-4497-8c6b-b5c2f6fcbbf3,393211,76m6v05NCeX8x2Wr5tKRE3,75007,CORRECT,"{""type"": ""DND_PAIRS"", ""input"": [""<p>Kleiner Ma...",2021-05-21 08:22:19.391,2021-05-21 08:22:55.366000128,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,2021,5,4,8,2021-05-17/2021-05-23,1.0,1,0,0,35.975


## 3. User-relative week index


In [6]:
first_activity = (
    transactions_clean.groupby("user_id", as_index=False)["start_time"]
    .min()
    .rename(columns={"start_time": "first_activity_time"})
)
events_clean = events_clean.merge(first_activity, on="user_id", how="left")
transactions_clean = transactions_clean.merge(first_activity, on="user_id", how="left")

events_clean["relative_week"] = (
    (events_clean["event_date"] - events_clean["first_activity_time"]).dt.days // 7
).astype("Int64")
transactions_clean["relative_week"] = (
    (transactions_clean["start_time"] - transactions_clean["first_activity_time"]).dt.days // 7
).astype("Int64")


## 4. Slice weeks 0–2 + identify the label cohort


In [7]:
ev_early = events_clean[events_clean["relative_week"].between(0, 2)].copy()
tx_early = transactions_clean[transactions_clean["relative_week"].between(0, 2)].copy()

late_event_users = set(
    events_clean.loc[events_clean["relative_week"] >= 3, "user_id"].unique()
)

# Free the full-history frames now that we've extracted what we need.
del events_clean, transactions_clean
gc.collect()

print(f"ev_early: {len(ev_early):,} events,       {ev_early['user_id'].nunique():,} users")
print(f"tx_early: {len(tx_early):,} transactions, {tx_early['user_id'].nunique():,} users")
print(f"late_event_users (relative_week >= 3): {len(late_event_users):,}")

ev_early: 4,161,449 events,       22,466 users
tx_early: 880,598 transactions, 22,470 users
late_event_users (relative_week >= 3): 13,655


## 5. Feature engineering — behavioural features


### 5.1 Event aggregates

Pure-event features over weeks 0–2: counts, distinct active days, time-of-day stats (`mean_hour` baseline, `std_hour` is the Käser-style mean+std twin).

In [8]:
def event_aggregates(ev_early: pd.DataFrame) -> pd.DataFrame:
    return (
        ev_early.groupby("user_id")
        .agg(
            total_events_0_2=("event_id", "count"),
            n_active_days_0_2=("date", "nunique"),
            n_click_events_0_2=("event_type", lambda x: (x.astype(str) == "CLICK").sum()),
            n_view_events_0_2=("event_type", lambda x: (x.astype(str) == "VIEW").sum()),
            n_sessions_0_2=("session_id", "nunique"),
            n_topics_event_0_2=("topic_id", "nunique"),
            mean_hour_0_2=("hour", "mean"),
            std_hour_0_2=("hour", "std"),
        )
        .reset_index()
    )

events_feat = event_aggregates(ev_early)
print(f"events_feat: {events_feat.shape}")
events_feat.head()

events_feat: (22466, 9)


,user_id,total_events_0_2,n_active_days_0_2,n_click_events_0_2,n_view_events_0_2,n_sessions_0_2,n_topics_event_0_2,mean_hour_0_2,std_hour_0_2
0,387604,2,2,0,2,0,0,9.000000,5.656854
1,387605,12,1,7,5,2,2,6.000000,0.000000
2,387608,101,2,35,66,2,1,8.603960,1.225391
3,387613,11,1,3,8,1,1,12.000000,0.000000
4,387615,348,6,76,272,2,1,9.784483,2.961246


### 5.2 Transaction aggregates

Question-answer outcomes over weeks 0–2: counts, correctness, evaluation score (mean and std), response time (mean and std), distinct documents/topics, plus `retry_ratio = total_transactions / unique_documents` (a wheel-spinning proxy from Beck & Gong 2013).

In [9]:
def transaction_aggregates(tx_early: pd.DataFrame) -> pd.DataFrame:
    tx_agg = (
        tx_early.groupby("user_id")
        .agg(
            total_transactions_0_2=("transaction_id", "count"),
            correct_rate_0_2=("is_correct", "mean"),
            partial_rate_0_2=("is_partial", "mean"),
            mean_evaluation_score_0_2=("evaluation_score", "mean"),
            std_evaluation_score_0_2=("evaluation_score", "std"),
            avg_response_time_0_2=("response_time_sec", "mean"),
            std_response_time_0_2=("response_time_sec", "std"),
            n_documents_0_2=("document_id", "nunique"),
            n_topics_transaction_0_2=("topic_id", "nunique"),
        )
        .reset_index()
    )
    tx_agg["retry_ratio_0_2"] = (
        tx_agg["total_transactions_0_2"] / tx_agg["n_documents_0_2"].replace(0, np.nan)
    )
    return tx_agg

transactions_feat = transaction_aggregates(tx_early)
print(f"transactions_feat: {transactions_feat.shape}")
transactions_feat.head()

transactions_feat: (22470, 11)


,user_id,total_transactions_0_2,correct_rate_0_2,partial_rate_0_2,mean_evaluation_score_0_2,std_evaluation_score_0_2,avg_response_time_0_2,std_response_time_0_2,n_documents_0_2,n_topics_transaction_0_2,retry_ratio_0_2
0,387604,2,0.000000,0.000000,0.000000,0.000000,NaN,NaN,1,0,2.000000
1,387605,5,0.400000,0.200000,0.500000,0.500000,39.957000,8.813259,5,2,1.000000
2,387608,34,0.382353,0.058824,0.411765,0.484152,26.930133,17.598100,15,1,2.266667
3,387613,2,0.500000,0.000000,0.500000,0.707107,24.976000,NaN,2,1,1.000000
4,387615,37,0.270270,0.081081,0.310811,0.446458,46.730929,47.616260,19,1,1.947368


### 5.3 Session shape — Chen & Cui 2020

Directly label-informative for a gap-based target ("any event in week ≥ 3"): widening between-session gaps in weeks 0–2 are an early dabbler signal.

In [10]:
def session_features(ev_early: pd.DataFrame) -> pd.DataFrame:
    sess_ev = ev_early.dropna(subset=["session_id"])
    sess = (
        sess_ev.groupby(["user_id", "session_id"])
        .agg(start=("event_date", "min"), end=("event_date", "max"))
        .reset_index()
    )
    sess["duration_sec"] = (sess["end"] - sess["start"]).dt.total_seconds()

    duration = (
        sess.groupby("user_id")
        .agg(
            session_duration_mean_0_2=("duration_sec", "mean"),
            session_duration_std_0_2=("duration_sec", "std"),
        )
        .reset_index()
    )

    sess = sess.sort_values(["user_id", "start"])
    sess["prev_end"] = sess.groupby("user_id")["end"].shift(1)
    sess["gap_sec"] = (
        (sess["start"] - sess["prev_end"]).dt.total_seconds().clip(lower=0)
    )
    gaps = (
        sess.groupby("user_id")
        .agg(time_between_sessions_mean_0_2=("gap_sec", "mean"))
        .reset_index()
    )
    return duration.merge(gaps, on="user_id", how="outer")

sessions_feat = session_features(ev_early)
print(f"sessions_feat: {sessions_feat.shape}")
sessions_feat.head()

sessions_feat: (17927, 4)


,user_id,session_duration_mean_0_2,session_duration_std_0_2,time_between_sessions_mean_0_2
0,387605,67.9890,91.349711,11.09300
1,387608,268.7685,140.867692,213.76500
2,387613,40.1110,NaN,NaN
3,387615,428.2360,176.792252,14.43900
4,387643,99804.3882,222446.474578,283988.01675


### 5.4 Review rate — Beck & Gong 2013

`review_rate_0_2 = REVIEW_TASK / SUBMIT_ANSWER` — how often the user reviews their answer relative to submitting one. 

In [11]:
def review_rate_feature(ev_early: pd.DataFrame) -> pd.DataFrame:
    counts = (
        ev_early.groupby(["user_id", "action"])
        .size()
        .unstack(fill_value=0)
    )
    for col in ["REVIEW_TASK", "SUBMIT_ANSWER"]:
        if col not in counts.columns:
            counts[col] = 0
    counts = counts.reset_index()
    counts["review_rate_0_2"] = (
        counts["REVIEW_TASK"] / counts["SUBMIT_ANSWER"].replace(0, np.nan)
    )
    return counts[["user_id", "review_rate_0_2"]]

review_feat = review_rate_feature(ev_early)
print(f"review_feat: {review_feat.shape}")
review_feat.head()

review_feat: (22466, 2)


action,user_id,review_rate_0_2
0,387604,NaN
1,387605,0.000000
2,387608,1.000000
3,387613,2.000000
4,387615,1.428571


## 6. Merge feature groups


In [12]:
behavioural = (
    events_feat
    .merge(transactions_feat, on="user_id", how="outer")
    .merge(sessions_feat, on="user_id", how="outer")
    .merge(review_feat, on="user_id", how="outer")
)

# Keep only users observed in weeks 0-2.
behavioural = behavioural[behavioural["total_events_0_2"].notna()].copy()

# Defensive: ensure every behavioural column exists.
for col in STAGE1_FEATURE_COLS:
    if col not in behavioural.columns:
        behavioural[col] = 0.0
behavioural[STAGE1_FEATURE_COLS] = behavioural[STAGE1_FEATURE_COLS].fillna(0.0)

print(f"behavioural: {behavioural.shape}  ({len(STAGE1_FEATURE_COLS)} feature columns)")
behavioural.head()

behavioural: (22466, 23)  (22 feature columns)


,user_id,total_events_0_2,n_active_days_0_2,n_click_events_0_2,n_view_events_0_2,n_sessions_0_2,n_topics_event_0_2,mean_hour_0_2,std_hour_0_2,total_transactions_0_2,...,std_evaluation_score_0_2,avg_response_time_0_2,std_response_time_0_2,n_documents_0_2,n_topics_transaction_0_2,retry_ratio_0_2,session_duration_mean_0_2,session_duration_std_0_2,time_between_sessions_mean_0_2,review_rate_0_2
0,387604,2.0,2.0,0.0,2.0,0.0,0.0,9.000000,5.656854,2,...,0.000000,0.000000,0.000000,1,0,2.000000,0.0000,0.000000,0.000,0.000000
1,387605,12.0,1.0,7.0,5.0,2.0,2.0,6.000000,0.000000,5,...,0.500000,39.957000,8.813259,5,2,1.000000,67.9890,91.349711,11.093,0.000000
2,387608,101.0,2.0,35.0,66.0,2.0,1.0,8.603960,1.225391,34,...,0.484152,26.930133,17.598100,15,1,2.266667,268.7685,140.867692,213.765,1.000000
3,387613,11.0,1.0,3.0,8.0,1.0,1.0,12.000000,0.000000,2,...,0.707107,24.976000,0.000000,2,1,1.000000,40.1110,0.000000,0.000,2.000000
4,387615,348.0,6.0,76.0,272.0,2.0,1.0,9.784483,2.961246,37,...,0.446458,46.730929,47.616260,19,1,1.947368,428.2360,176.792252,14.439,1.428571


## 7. Demographics — 66 features

Total: 2 numeric + ~64 one-hot = ~66 columns (the exact count depends on the cantons and school types actually present).

In [13]:
def demographic_features(users: pd.DataFrame) -> pd.DataFrame:
    u = users[["user_id", "gender", "canton", "class_level", "study"]].copy()

    u["gender"] = u["gender"].fillna("Unknown").astype(str)
    u["canton"] = u["canton"].fillna("Unknown").astype(str)
    u["class_level"] = u["class_level"].fillna("Unknown").astype(str)

    u["school_type"] = u["class_level"].str.split(" - ").str[0].fillna("Unknown")
    year = u["class_level"].str.extract(r"(\d+)\. Jahr")[0]
    u["class_year"] = pd.to_numeric(year, errors="coerce").fillna(0).astype(int)

    u["study"] = u["study"].astype(int)

    dummies = pd.get_dummies(
        u[["gender", "canton", "school_type"]],
        prefix=["gender", "canton", "school"],
    ).astype(int)

    return pd.concat(
        [
            u[["user_id", "study", "class_year"]].reset_index(drop=True),
            dummies.reset_index(drop=True),
        ],
        axis=1,
    )

demographics_feat = demographic_features(users_clean)
demo_cols = [c for c in demographics_feat.columns if c != "user_id"]
print(f"demographics_feat: {demographics_feat.shape}  ({len(demo_cols)} feature columns)")
demographics_feat.head()

demographics_feat: (30929, 67)  (66 feature columns)


,user_id,study,class_year,gender_FEMALE,gender_MALE,gender_Other,gender_Unknown,canton_AG,canton_AI,canton_AR,...,school_Maturität für Erwachsene,school_Passerelle,school_Passerelle BM/FM,school_Sekundarschule P,school_UG,school_Unknown,school_Vorkurs PH für Berufsleute,school_Vorkurs Pädagogik,school_WMS,school_andere
0,387604,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,387605,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,387608,1,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,387613,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,387615,0,3,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 8. Merge demographics + compute label

In [14]:
df = behavioural.merge(demographics_feat, on="user_id", how="left")
df[demo_cols] = df[demo_cols].fillna(0)

df["came_back"] = df["user_id"].isin(late_event_users).astype(int)

print(f"df: {df.shape}  (behavioural {len(STAGE1_FEATURE_COLS)} + demographic {len(demo_cols)} = {len(STAGE1_FEATURE_COLS) + len(demo_cols)} features + 1 label)")
print(f"came_back rate: {df['came_back'].mean():.3f}")
print("\nLabel counts:")
print(df["came_back"].value_counts().rename({0: "left (dabbler)", 1: "came_back"}))
df.head()

df: (22466, 90)  (behavioural 22 + demographic 66 = 88 features + 1 label)
came_back rate: 0.608

Label counts:
came_back
came_back         13652
left (dabbler)     8814
Name: count, dtype: int64


,user_id,total_events_0_2,n_active_days_0_2,n_click_events_0_2,n_view_events_0_2,n_sessions_0_2,n_topics_event_0_2,mean_hour_0_2,std_hour_0_2,total_transactions_0_2,...,school_Passerelle,school_Passerelle BM/FM,school_Sekundarschule P,school_UG,school_Unknown,school_Vorkurs PH für Berufsleute,school_Vorkurs Pädagogik,school_WMS,school_andere,came_back
0,387604,2.0,2.0,0.0,2.0,0.0,0.0,9.000000,5.656854,2,...,0,0,0,0,1,0,0,0,0,1
1,387605,12.0,1.0,7.0,5.0,2.0,2.0,6.000000,0.000000,5,...,0,0,0,0,1,0,0,0,0,1
2,387608,101.0,2.0,35.0,66.0,2.0,1.0,8.603960,1.225391,34,...,0,0,0,0,1,0,0,0,0,1
3,387613,11.0,1.0,3.0,8.0,1.0,1.0,12.000000,0.000000,2,...,0,0,0,0,1,0,0,0,0,1
4,387615,348.0,6.0,76.0,272.0,2.0,1.0,9.784483,2.961246,37,...,0,0,0,0,0,0,0,0,0,1


## 9. Save


In [15]:
feature_cols = STAGE1_FEATURE_COLS + demo_cols
out_cols = ["user_id", *feature_cols, "came_back"]

OUT_DATASET.parent.mkdir(parents=True, exist_ok=True)
df[out_cols].to_csv(OUT_DATASET, index=False, compression="gzip")